# Ground-detector geometry: original kernels and JAX

Held, generated inputs demonstrate the arithmetic boundaries. Every original-route check compares dtype, shape, and bytes. CPU and JAX use the same detector metadata. Set `DEVICE = "cuda"` to repeat on an available GPU; output differences depend on build/device.

In [1]:
import warnings
warnings.filterwarnings("ignore", message="Wswiglal-redir-stdio:.*")
import importlib.metadata
import numpy as np
import jax
import jax.numpy as jnp
from pycbc.detector import Detector, NetworkGeometry
from pycbc.detector.ground import single_arm_frequency_response
from pycbc.scheme import JAXScheme
jax.config.update("jax_enable_x64", True)
DEVICE = "cpu"
for name in ("numpy", "jax", "astropy"):
    print(name, importlib.metadata.version(name))
print("device platform:", jax.devices()[0].platform)
TIME = 1126259462.0

def exact(actual, expected):
    if isinstance(expected, tuple):
        for a, e in zip(actual, expected):
            exact(a, e)
        return
    a, e = np.asarray(actual), np.asarray(expected)
    assert a.dtype == e.dtype and a.shape == e.shape
    assert a.tobytes() == e.tobytes()

def measure(actual, expected):
    a, e = np.asarray(actual), np.asarray(expected)
    return {"different values": int(np.count_nonzero(a != e)),
            "max absolute difference": float(np.max(np.abs(a-e)))}

def evaluate(function, inputs, references=(), **kwargs):
    with JAXScheme(DEVICE, reference_operations=references) as context:
        values = tuple(jax.device_put(v, context.jax_device) if isinstance(v, np.ndarray) else v for v in inputs)
        options = {name: jax.device_put(value, context.jax_device) if isinstance(value, np.ndarray) else value for name, value in kwargs.items()}
        return function(*values, **options)

rng = np.random.default_rng(913)
ra = rng.uniform(0, 2*np.pi, 129)
dec = rng.uniform(-np.pi/2, np.pi/2, 129)
pol = rng.uniform(0, np.pi, 129)
times = TIME + np.arange(129, dtype=np.float64)*.125
detector = Detector("H1")
inputs = (ra, dec, pol, times)

numpy 2.5.2
jax 0.11.1
astropy 8.0.1
device platform: cpu


## Independent antenna, delay, and clock controls

The original clock adds a phase and takes modulo $2\pi$. Default device geometry shares a basis using angle-addition identities. Native controls are concrete, deliberately slow, and not differentiable.

In [2]:
cpu_antenna = detector.antenna_pattern(*inputs)
cpu_delay = detector.time_delay_from_earth_center(ra, dec, times)
original_antenna = evaluate(detector.antenna_pattern, inputs, ("detector.Detector.antenna_pattern",))
original_delay = evaluate(detector.time_delay_from_earth_center, (ra, dec, times), ("detector.Detector.time_delay_from_location",))
exact(original_antenna, cpu_antenna)
exact(original_delay, cpu_delay)
original_clock = evaluate(detector.gmst_estimate, (times,), ("detector.Detector.gmst_estimate",))
exact(original_clock, detector.gmst_estimate(times))
default_antenna = evaluate(detector.antenna_pattern, inputs)
default_delay = evaluate(detector.time_delay_from_earth_center, (ra, dec, times))
print("antenna:", [measure(a,e) for a,e in zip(default_antenna,cpu_antenna)])
print("delay:", measure(default_delay,cpu_delay))
# Original modulo, direct unmodded cosine, and angle addition use the same NumPy library.
long_times = TIME + np.linspace(0, 3*detector.sday, 257)
offsets = (long_times-detector.reference_time)/detector.sday*(2*np.pi)
start = detector.gmst_reference - .3
original_cosine = np.cos(detector.gmst_estimate(long_times)-.3)
unmodded_cosine = np.cos(start+offsets)
split_cosine = np.cos(start)*np.cos(offsets)-np.sin(start)*np.sin(offsets)
print("modulo vs direct:", measure(unmodded_cosine, original_cosine))
print("modulo vs angle addition:", measure(split_cosine, original_cosine))

antenna: [{'different values': 91, 'max absolute difference': 4.718447854656915e-16}, {'different values': 93, 'max absolute difference': 4.440892098500626e-16}]
delay: {'different values': 53, 'max absolute difference': 6.938893903907228e-18}
modulo vs direct: {'different values': 214, 'max absolute difference': 2.1718737919229625e-15}
modulo vs angle addition: {'different values': 221, 'max absolute difference': 9.43689570931383e-16}


## Freeze the basis to isolate contractions

NumPy `dot` and its elementwise reductions are compared with XLA `dot` and reductions. A differing result here cannot come from GMST or trigonometry: the operands are identical. This does not identify a compiler instruction sequence.

In [3]:
gha = detector.gmst_estimate(times)-ra
cg, sg, cd, sd, cp, sp = np.cos(gha),np.sin(gha),np.cos(dec),np.sin(dec),np.cos(pol),np.sin(pol)
x = np.stack((-cp*sg-sp*cg*sd, -cp*cg+sp*sg*sd, sp*cd))
y = np.stack((sp*sg-cp*cg*sd, sp*cg+cp*sg*sd, cp*cd))
def cpu_contraction(response,x,y):
    dx,dy=response.dot(x),response.dot(y)
    return (x*dx-y*dy).sum(axis=0),(x*dy+y*dx).sum(axis=0)
def device_contraction(response,x,y):
    dx,dy=jnp.dot(response,x),jnp.dot(response,y)
    return jnp.sum(x*dx-y*dy,axis=0),jnp.sum(x*dy+y*dx,axis=0)
held_cpu=cpu_contraction(detector.response,x,y)
held_device=evaluate(jax.jit(device_contraction),(detector.response,x,y))
print("fixed-basis contraction:",[measure(a,e) for a,e in zip(held_device,held_cpu)])

fixed-basis contraction: [{'different values': 34, 'max absolute difference': 1.1102230246251565e-16}, {'different values': 33, 'max absolute difference': 1.1102230246251565e-16}]


## Compose independent controls

The distance-scale control executes the original effective-distance body with the already computed antenna factors. Both flags together restore the complete original result. Combined/network methods independently honor original antenna and delay controls. Response time is exactly the supplied time; use `arrival_time` first when the model calls for response at detector arrival.

In [4]:
distance=np.linspace(100,500,len(ra))
distance_inputs=(distance,ra,dec,pol,times,.7)
cpu_distance=detector.effective_distance(*distance_inputs)
antenna_only=evaluate(detector.effective_distance,distance_inputs,("detector.Detector.antenna_pattern",))
composed=evaluate(detector.effective_distance,distance_inputs,("detector.Detector.antenna_pattern","detector.Detector.effective_distance_scale"))
whole=evaluate(detector.effective_distance,distance_inputs,("detector.Detector.effective_distance",))
exact(composed,cpu_distance)
exact(whole,cpu_distance)
print("remaining after antenna control:",measure(antenna_only,cpu_distance))
network=NetworkGeometry(["H1","L1","V1"])
cpu_network=network.antenna_pattern_and_time_delay(*inputs)
original_network=evaluate(network.antenna_pattern_and_time_delay,inputs,("detector.Detector.antenna_pattern","detector.Detector.time_delay_from_location"))
exact(original_network,cpu_network)
cpu_arrival=detector.arrival_time(times,ra,dec)
original_arrival=evaluate(detector.arrival_time,(times,ra,dec),("detector.Detector.time_delay_from_location",))
exact(original_arrival,cpu_arrival)
print("distance, network, and arrival composition: exact")

remaining after antenna control: {'different values': 0, 'max absolute difference': 0.0}
distance, network, and arrival composition: exact


## Finite-arm cancellation

Let $p=2\pi fL/c$. The original complex-exponential difference and the JAX phase-times-sinc expression are equivalent for nonzero phase. When $	heta$ is small, `1-cos(theta)` loses its small real part; `2*sin(theta/2)**2` retains it. The original transfer function is undefined at zero frequency, and its JAX counterpart retains that NaN. Antenna frequency zero uses the static detector matrix.

In [5]:
frequency=np.array([0.,1e-6,10.,1000.])
with np.errstate(all="ignore"):
    cpu_arm=single_arm_frequency_response(frequency,.1,4000.)
    native_arm=evaluate(single_arm_frequency_response,(frequency,.1,4000.),("detector.single_arm_frequency_response",))
    default_arm=evaluate(single_arm_frequency_response,(frequency,.1,4000.))
exact(native_arm,cpu_arm)
assert np.isnan(cpu_arm[0]) and np.isnan(np.asarray(default_arm)[0])
print("original imaginary component:",cpu_arm[1:].imag)
print("JAX imaginary component:",np.asarray(default_arm)[1:].imag)
theta=1e-10
print("1-cos(theta):",1-np.cos(theta),"stable identity:",2*np.sin(theta/2)**2)
assert 1-np.cos(theta)==0 and 2*np.sin(theta/2)**2>0
# The original scalar antenna method is replayed for a new frequency-grid API.
positive_frequency=frequency[1:]
scalar_rows=[detector.antenna_pattern(.3,-.2,.1,TIME,frequency=f) for f in positive_frequency]
expected=tuple(np.asarray([row[i] for row in scalar_rows]) for i in range(2))
actual=evaluate(detector.antenna_pattern,(.3,-.2,.1,TIME), ("detector.Detector.antenna_pattern",),frequency=positive_frequency)
exact(actual,expected)

original imaginary component: [-8.38338009e-11 -7.96420928e-04 -7.94621081e-02]
JAX imaginary component: [-7.96421108e-11 -7.96420928e-04 -7.94621081e-02]
1-cos(theta): 0.0 stable identity: 5.0000000000000005e-21


## Precision is part of the input

These two time arrays are different inputs. Float32 has already discarded subsecond spacing before either implementation runs. Native validation must compare the same rounded array, rather than claim it recovers the original float64 times.

In [6]:
precise_times=TIME+np.arange(8,dtype=np.float64)*.125
rounded_times=precise_times.astype(np.float32)
print("distinct float64 times:",len(np.unique(precise_times)))
print("distinct float32 times:",len(np.unique(rounded_times)))
assert len(np.unique(rounded_times))<len(np.unique(precise_times))
values=(np.full(8,.3),np.full(8,-.2),rounded_times)
cpu_rounded=detector.time_delay_from_earth_center(*values)
native_rounded=evaluate(detector.time_delay_from_earth_center,values,("detector.Detector.time_delay_from_location",))
exact(native_rounded,cpu_rounded)

distinct float64 times: 8
distinct float32 times: 1


## New grids and differentiation

Where the original NumPy dot/broadcast shape is supported, JAX retains those axes. Additional grids use component-axis contractions and scalar native replay; the example below is an API extension, not a rounding tolerance. Original controls reject traced inputs.

In [7]:
grid=np.linspace(.3,1.2,6).reshape(2,3)
try:
    detector.antenna_pattern(grid,-.2,.1,TIME)
except ValueError:
    pass
else:
    raise AssertionError("Expected the original array-shape restriction")
rows=[detector.antenna_pattern(value,-.2,.1,TIME) for value in grid.flat]
expected=tuple(np.asarray([row[i] for row in rows]).reshape(grid.shape) for i in range(2))
actual=evaluate(detector.antenna_pattern,(grid,-.2,.1,TIME),("detector.Detector.antenna_pattern",))
exact(actual,expected)
with JAXScheme(DEVICE):
    gradient=jax.grad(lambda angle:jnp.sum(detector.antenna_pattern(angle,-.2,.1,TIME)[0]))(jnp.asarray(ra))
    assert bool(jnp.all(jnp.isfinite(gradient)))
print("extended-grid native replay and default gradient: passed")

extended-grid native replay and default gradient: passed
